### 01b - Multi-Shadow Model Training

Trains N architecturally-identical shadow models on disjoint, stratified subsets of the shadow training data.

Each shadow model *i* trains on its own slice of `shadow_train` and is evaluated on the shared `shadow_test`. The attacker controls all of this data, so membership labels are known for every row used to train the attack classifier later.

##### Configurable parameters
- `N_SHADOWS` – number of shadow models (default 5)
- `RANDOM_STATE` – reproducibility seed

##### Inputs
- `data/external/BaselineDataSplits/shadow_{train,test}.csv`
- `data/external/clinicalbert/{x1,x2,y}_shadow_{train,test}.npy`

##### Outputs
- `outputs/models/shadow_encoder_{i}.h5`
- `outputs/models/shadow_clf_{i}.h5`
- `outputs/models/shadow_{x1,x2,y}_train_{i}.npy` (cached subset embeddings)

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model, Input
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [ ]:
# ==========================
# CONFIG
# ==========================
N_SHADOWS = 10
RANDOM_STATE = 42
SHADOW_TRAIN_CSV = 'data/external/BaselineDataSplits/shadow_train.csv'
SHADOW_TEST_CSV  = 'data/external/BaselineDataSplits/shadow_test.csv'
EMB_DIR = 'data/external/clinicalbert'
OUT_DIR = 'outputs/models'
os.makedirs(OUT_DIR, exist_ok=True)

# Architecture / training hyperparameters (keep identical to target)
SA_EPOCHS = 30
SA_BATCH_SIZE = 256
CLF_EPOCHS = 20
CLF_BATCH_SIZE = 256
MARGIN = 2.5
ALPHA = 1.0
THRESHOLD = 1.0


In [ ]:
# ==========================
# Load CSVs and full embeddings
# ==========================
train_df = pd.read_csv(SHADOW_TRAIN_CSV)
test_df  = pd.read_csv(SHADOW_TEST_CSV)

x1_train_full = np.load(f'{EMB_DIR}/x1_shadow_train.npy')
x2_train_full = np.load(f'{EMB_DIR}/x2_shadow_train.npy')
y_train_full  = np.load(f'{EMB_DIR}/y_shadow_train.npy')

x1_test = np.load(f'{EMB_DIR}/x1_shadow_test.npy')
x2_test = np.load(f'{EMB_DIR}/x2_shadow_test.npy')
y_test  = np.load(f'{EMB_DIR}/y_shadow_test.npy')

print(f"Full shadow train: {len(train_df)} rows, labels: {train_df['label'].value_counts().to_dict()}")
print(f"Shadow test:       {len(test_df)} rows, labels: {test_df['label'].value_counts().to_dict()}")


In [ ]:
# ==========================
# Stratified disjoint split of shadow_train into N subsets
# ==========================
def stratified_disjoint_split(df, n_splits, random_state=42):
    rng = np.random.RandomState(random_state)
    class_splits = {}
    for label, group in df.groupby('label'):
        idx = group.index.values.copy()
        rng.shuffle(idx)
        class_splits[label] = np.array_split(idx, n_splits)
    fold_indices = []
    for i in range(n_splits):
        fold_idx = np.concatenate([
            class_splits[label][i] for label in sorted(class_splits)
        ])
        fold_idx = rng.permutation(fold_idx)
        fold_indices.append(fold_idx)
    return fold_indices

fold_indices = stratified_disjoint_split(train_df, N_SHADOWS, RANDOM_STATE)
for i, idx in enumerate(fold_indices):
    sub = train_df.loc[idx]
    print(f"Shadow {i}: {len(sub)} rows, labels: {sub['label'].value_counts().to_dict()}")


In [ ]:
# ==========================
# Model architecture helpers (identical to target / single-shadow notebooks)
# ==========================
def build_siamese_autoencoder(embedding_dim):
    encoder_input = Input(shape=(embedding_dim,))
    x = layers.Dense(50, activity_regularizer=regularizers.l1(0.01))(encoder_input)
    x = layers.LeakyReLU(alpha=0.01)(x)
    encoder_output = layers.Dense(embedding_dim)(x)
    encoder = Model(encoder_input, encoder_output)

    decoder_input = Input(shape=(embedding_dim,))
    decoder_output = layers.Dense(embedding_dim, activation='sigmoid')(decoder_input)
    decoder = Model(decoder_input, decoder_output)

    input1 = Input(shape=(embedding_dim,))
    input2 = Input(shape=(embedding_dim,))
    encoded1 = encoder(input1)
    encoded2 = encoder(input2)
    recon1 = decoder(encoded1)
    recon2 = decoder(encoded2)
    merged_output = layers.Concatenate()([encoded1, encoded2, recon1, recon2])
    model = Model(inputs=[input1, input2], outputs=merged_output)
    return model, encoder

def hybrid_classification_loss(margin=2.5, alpha=1.0):
    def loss_fn(y_true, y_pred):
        emb_dim = tf.shape(y_pred)[1] // 4
        encoded1 = y_pred[:, :emb_dim]
        encoded2 = y_pred[:, emb_dim:2*emb_dim]
        recon1 = y_pred[:, 2*emb_dim:3*emb_dim]
        recon2 = y_pred[:, 3*emb_dim:]
        distances = tf.norm(encoded1 - encoded2, axis=1)
        y_true = tf.cast(y_true, tf.float32)
        contrastive_loss = y_true * tf.square(distances) + (1 - y_true) * tf.square(tf.maximum(margin - distances, 0))
        recon_loss1 = tf.reduce_mean(tf.square(encoded1 - recon1), axis=1)
        recon_loss2 = tf.reduce_mean(tf.square(encoded2 - recon2), axis=1)
        recon_loss = 0.5 * (recon_loss1 + recon_loss2)
        return tf.reduce_mean(alpha * recon_loss + contrastive_loss)
    return loss_fn

def build_classifier(input_dim):
    input_diff = Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(input_diff)
    x = layers.Dense(32, activation='relu')(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    clf = Model(inputs=input_diff, outputs=output)
    clf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return clf


In [ ]:
# ==========================
# Train each shadow model
# ==========================
embedding_dim = x1_train_full.shape[1]
summary = []

for i in range(N_SHADOWS):
    print(f"\n{'='*50}")
    print(f"Training shadow model {i}/{N_SHADOWS}")
    print(f"{'='*50}")

    # Slice embeddings for this shadow's train subset
    idx = fold_indices[i]
    x1_tr = x1_train_full[idx]
    x2_tr = x2_train_full[idx]
    y_tr  = y_train_full[idx]

    # Cache subset embeddings for later feature extraction
    np.save(f'{OUT_DIR}/shadow_x1_train_{i}.npy', x1_tr)
    np.save(f'{OUT_DIR}/shadow_x2_train_{i}.npy', x2_tr)
    np.save(f'{OUT_DIR}/shadow_y_train_{i}.npy',  y_tr)

    # ---- Siamese Autoencoder ----
    sa_model, encoder = build_siamese_autoencoder(embedding_dim)
    sa_model.compile(optimizer='adam', loss=hybrid_classification_loss(margin=MARGIN, alpha=ALPHA))
    sa_model.fit([x1_tr, x2_tr], y_tr, epochs=SA_EPOCHS, batch_size=SA_BATCH_SIZE,
                 validation_split=0.1, verbose=2)

    # ---- Classification Head ----
    enc1_tr = encoder.predict(x1_tr, verbose=0)
    enc2_tr = encoder.predict(x2_tr, verbose=0)
    diff_tr = np.abs(enc1_tr - enc2_tr)

    enc1_te = encoder.predict(x1_test, verbose=0)
    enc2_te = encoder.predict(x2_test, verbose=0)
    diff_te = np.abs(enc1_te - enc2_te)

    clf_model = build_classifier(diff_tr.shape[1])
    clf_model.fit(diff_tr, y_tr, epochs=CLF_EPOCHS, batch_size=CLF_BATCH_SIZE,
                  validation_split=0.1, verbose=2)

    # ---- Evaluate ----
    y_prob_te = clf_model.predict(diff_te, verbose=0).flatten()
    y_pred_te = (y_prob_te > 0.5).astype(int)
    acc = accuracy_score(y_test, y_pred_te)
    f1  = f1_score(y_test, y_pred_te)

    y_prob_tr = clf_model.predict(diff_tr, verbose=0).flatten()
    y_pred_tr = (y_prob_tr > 0.5).astype(int)
    acc_tr = accuracy_score(y_tr, y_pred_tr)
    f1_tr  = f1_score(y_tr, y_pred_tr)

    print(f"Shadow {i} — Train acc: {acc_tr:.4f}, Train F1: {f1_tr:.4f} | Test acc: {acc:.4f}, Test F1: {f1:.4f}")
    summary.append({'shadow': i, 'train_acc': acc_tr, 'train_f1': f1_tr,
                    'test_acc': acc, 'test_f1': f1})

    # ---- Save ----
    encoder.save(f'{OUT_DIR}/shadow_encoder_{i}.h5')
    clf_model.save(f'{OUT_DIR}/shadow_clf_{i}.h5')

    # Save this model's predictions for convenience
    np.save(f'{OUT_DIR}/shadow_train_probs_{i}.npy', y_prob_tr)
    np.save(f'{OUT_DIR}/shadow_test_probs_{i}.npy',  y_prob_te)

print("\nAll shadow models trained.\nSummary:")
print(pd.DataFrame(summary))
